# Notebook guide

This notebook explores and compares diversification-oriented ETF selection methods.

## Inputs
- data/raw/volume_screen.csv
- data/raw/daily_close_volume_screened_2016_2025.parquet

## What to verify
- Universe filter impact and survivor counts
- Diversity diagnostics (distance/correlation structure)
- Stability and overlap across selection methods

Execute cells in order so all matrices and plots share the same filtered universe.

# Explore Correlation Selection Methods

This notebook focuses on the selection methods compared interactively here. Shared data preparation happens once, then each included method gets a Markdown explanation and a dedicated execution cell.

Every selector below treats `VOO` and `VEA` as hard portfolio anchors. Those two ETFs are always included first, and each method only decides how to fill the remaining slots with low-correlation names.

The notebook also applies the correlation-stage eligibility screens before any method runs: first `start_date <= 2018-01-01`, then a cumulative log-return hurdle over the full 2018-01-01 to 2025-12-31 window. The shared setup cell includes `RETURN_FREQUENCY`, so you can switch between daily and weekly log returns without changing any method cell.



In [1]:
from __future__ import annotations
from pathlib import Path
import importlib
import sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "GUIDE_ROOT.md").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
for _candidate in [PROJECT_ROOT / "src", PROJECT_ROOT]:
    if str(_candidate) not in sys.path:
        sys.path.insert(0, str(_candidate))

import notebook_support as nb_support
import correlation_analysis.correlate_greedy as correlate_greedy
import correlation_analysis.correlate_kmedoids as correlate_kmedoids
import correlation_analysis.correlate_maxdiv as correlate_maxdiv
import correlation_analysis.correlate_utils as utils

nb_support = importlib.reload(nb_support)
correlate_greedy = importlib.reload(correlate_greedy)
correlate_kmedoids = importlib.reload(correlate_kmedoids)
correlate_maxdiv = importlib.reload(correlate_maxdiv)
utils = importlib.reload(utils)

PRICE_PARQUET = nb_support.PRICE_PARQUET
SCREEN_CSV = nb_support.SCREEN_CSV
bootstrap_notebook = nb_support.bootstrap_notebook
build_membership_frame = nb_support.build_membership_frame
build_overlap_matrix = nb_support.build_overlap_matrix
ensure_required_artifacts = nb_support.ensure_required_artifacts
load_local_price_data = nb_support.load_local_price_data

PROJECT_ROOT = bootstrap_notebook(PROJECT_ROOT)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 180)

TOP_N = 30
ANALYSIS_START = "20160101"
ANALYSIS_END = "20251231"
RETURN_FREQUENCY = "weekly"
MIN_TOTAL_RETURN_HURDLE = 0.2
MIN_AVERAGE_YEARLY_RETURN = 0.0
MIN_ANN_VOL = 0.1
MAX_ANN_VOL = 0.60
BLACKLIST_TICKERS = [
    "GLD",
    "ASHR",
    "ELV",
    "KSA",
    "EWZ",
    "UGL",
    "VNM",
    "ECH",
    "BAR",
    "SGOL",
    "PHYS",
    "OUNZ",
    "GDXJ",
]
# Required core holdings seeded before any method optimises
ANCHOR_TICKERS = ["VOO", "VEA"]
ANALYSIS_START, ANALYSIS_END = utils.resolve_analysis_window(
    ANALYSIS_START,
    ANALYSIS_END,
)

ensure_required_artifacts(
    {
        "volume screen": SCREEN_CSV,
        "daily parquet": PRICE_PARQUET,
    }
)

screened_candidates = utils.load_screened_universe()
candidates = utils.apply_start_date_filter(
    screened_candidates,
    start_date=ANALYSIS_START,
)
prices_long = load_local_price_data(["ticker", "date", "close_price"]).rename(
    columns={"ticker": "symbol"}
)
prices_long = prices_long[prices_long["symbol"].isin(candidates["ticker"])].copy()
prices_long["date"] = pd.to_datetime(prices_long["date"])
prices_long = prices_long[
    (prices_long["date"] >= pd.Timestamp(ANALYSIS_START))
    & (prices_long["date"] <= pd.Timestamp(ANALYSIS_END))
].copy()
log_ret, survivors = utils.build_return_matrix(
    prices_long,
    candidates,
    return_frequency=RETURN_FREQUENCY,
)

coverage_candidate_count = len(candidates)
coverage_survivor_count = len(survivors)
preselection_audit_rows: list[dict[str, object]] = []


def subset_survivors_to_tickers(
    filtered_log_ret: pd.DataFrame,
    filtered_survivors: pd.DataFrame,
    passing_tickers: list[str],
) -> tuple[pd.DataFrame, pd.DataFrame]:
    selected_log_ret = filtered_log_ret.loc[:, passing_tickers].copy()
    selected_survivors = filtered_survivors[
        filtered_survivors["ticker"].isin(passing_tickers)
    ].copy()
    selected_survivors = selected_survivors.sort_values(
        "vol_combined",
        ascending=False,
    ).reset_index(drop=True)
    return selected_log_ret, selected_survivors


def record_audit_step(
    step_name: str,
    threshold: str,
    count_before: int,
    count_after: int,
) -> None:
    preselection_audit_rows.append(
        {
            "step": step_name,
            "threshold": threshold,
            "count_before": count_before,
            "count_after": count_after,
            "removed_count": count_before - count_after,
        }
    )


record_audit_step(
    step_name="coverage",
    threshold=f">= {utils.MIN_COVERAGE:.0%} observed return rows",
    count_before=coverage_candidate_count,
    count_after=coverage_survivor_count,
)

current_log_ret = log_ret
current_survivors = survivors

count_before = len(current_survivors)
current_log_ret, current_survivors = utils.filter_blacklisted_tickers(
    current_log_ret,
    current_survivors,
    blacklist_tickers=BLACKLIST_TICKERS,
    anchor_tickers=ANCHOR_TICKERS,
)
record_audit_step(
    step_name="blacklist",
    threshold=", ".join(BLACKLIST_TICKERS),
    count_before=count_before,
    count_after=len(current_survivors),
)

count_before = len(current_survivors)
current_log_ret, current_survivors = utils.filter_min_total_return(
    current_log_ret,
    current_survivors,
    min_total_return=MIN_TOTAL_RETURN_HURDLE,
    start_date=ANALYSIS_START,
    end_date=ANALYSIS_END,
    anchor_tickers=ANCHOR_TICKERS,
)
record_audit_step(
    step_name="min_total_return",
    threshold=f">= {MIN_TOTAL_RETURN_HURDLE:.4f} cumulative log return",
    count_before=count_before,
    count_after=len(current_survivors),
)

count_before = len(current_survivors)
current_log_ret, current_survivors = utils.filter_min_average_yearly_return(
    current_log_ret,
    current_survivors,
    min_average_yearly_return=MIN_AVERAGE_YEARLY_RETURN,
    anchor_tickers=ANCHOR_TICKERS,
)
record_audit_step(
    step_name="min_average_yearly_return",
    threshold=f">= {MIN_AVERAGE_YEARLY_RETURN:.4f} average calendar-year log return",
    count_before=count_before,
    count_after=len(current_survivors),
)

ann_vols = utils.compute_annualized_volatility(
    current_log_ret,
    return_frequency=RETURN_FREQUENCY,
)
dropped_min_vol = ann_vols[
    (ann_vols < MIN_ANN_VOL) & (~ann_vols.index.isin(ANCHOR_TICKERS))
].sort_values()
passing_min_vol = [
    ticker for ticker in current_log_ret.columns if ticker not in dropped_min_vol.index
]
count_before = len(current_survivors)
current_log_ret, current_survivors = subset_survivors_to_tickers(
    current_log_ret,
    current_survivors,
    passing_min_vol,
)
utils.ensure_anchor_tickers(
    current_log_ret.columns.tolist(),
    "minimum-volatility-filtered survivor universe",
    ANCHOR_TICKERS,
)
record_audit_step(
    step_name="min_ann_vol",
    threshold=f">= {MIN_ANN_VOL:.1%}",
    count_before=count_before,
    count_after=len(current_survivors),
)

ann_vols = utils.compute_annualized_volatility(
    current_log_ret,
    return_frequency=RETURN_FREQUENCY,
)
dropped_max_vol = ann_vols[
    (ann_vols > MAX_ANN_VOL) & (~ann_vols.index.isin(ANCHOR_TICKERS))
].sort_values(ascending=False)
passing_max_vol = [
    ticker for ticker in current_log_ret.columns if ticker not in dropped_max_vol.index
]
count_before = len(current_survivors)
current_log_ret, current_survivors = subset_survivors_to_tickers(
    current_log_ret,
    current_survivors,
    passing_max_vol,
)
utils.ensure_anchor_tickers(
    current_log_ret.columns.tolist(),
    "maximum-volatility-filtered survivor universe",
    ANCHOR_TICKERS,
)
record_audit_step(
    step_name="max_ann_vol",
    threshold=f"<= {MAX_ANN_VOL:.0%}",
    count_before=count_before,
    count_after=len(current_survivors),
)

log_ret = current_log_ret
survivors = current_survivors
corr = utils.compute_spearman_corr(log_ret)
dist = utils.compute_distance_matrix(corr)
preselection_audit = pd.DataFrame(preselection_audit_rows)
panel_summary = pd.Series(
    {
        "top_n_per_method": TOP_N,
        "return_frequency": RETURN_FREQUENCY,
        "required_anchors": ", ".join(ANCHOR_TICKERS),
        "manual_blacklist": ", ".join(BLACKLIST_TICKERS),
        "analysis_start": ANALYSIS_START,
        "analysis_end": ANALYSIS_END,
        "history_start_cutoff": ANALYSIS_START,
        "history_end": ANALYSIS_END,
        "min_total_return_hurdle": f"{MIN_TOTAL_RETURN_HURDLE:.4f} cumulative log return",
        "min_average_yearly_return": f"{MIN_AVERAGE_YEARLY_RETURN:.4f} average calendar-year log return",
        "min_ann_vol": f"{MIN_ANN_VOL:.1%}",
        "max_ann_vol": f"{MAX_ANN_VOL:.0%}",
        "candidate_count_after_liquidity_screen": len(screened_candidates),
        "candidate_count_after_start_date_filter": len(candidates),
        "survivor_count_after_coverage_filter": coverage_survivor_count,
        "survivor_count_after_all_preselection_filters": len(survivors),
        "return_observations": len(log_ret),
        "start_date": str(log_ret.index.min().date()),
        "end_date": str(log_ret.index.max().date()),
    },
    name="value",
)
print("Shared return/correlation panel:")
display(panel_summary.to_frame())
print("Pre-selection audit:")
display(preselection_audit)

selection_results: dict[str, dict[str, object]] = {}


def plot_selected_corr(selected: list[str], title: str) -> None:
    selected_corr = corr.loc[selected, selected]
    fig, ax = plt.subplots(figsize=(8, 6), constrained_layout=True)
    image = ax.imshow(selected_corr.values, cmap="coolwarm", vmin=-1.0, vmax=1.0)
    ax.set_xticks(range(len(selected)), selected, rotation=35)
    ax.set_yticks(range(len(selected)), selected)
    ax.set_title(title)
    fig.colorbar(image, ax=ax, fraction=0.046)
    display(fig)
    plt.close(fig)


def summarize_selection(method_name: str, selected: list[str]) -> pd.DataFrame:
    top_selected = utils.build_anchor_first_selection(
        ranked_tickers=selected,
        available_tickers=selected,
        n_select=TOP_N,
        anchor_tickers=ANCHOR_TICKERS,
    )
    stats = utils.compute_stats(
        top_selected,
        log_ret,
        survivors,
        return_frequency=RETURN_FREQUENCY,
        anchor_tickers=ANCHOR_TICKERS,
    ).reset_index(drop=True)
    selection_results[method_name] = {"selected": top_selected, "stats": stats}
    print(f"{method_name} top-{TOP_N} tickers: {', '.join(top_selected)}")
    display(
        stats[
            [
                "ticker",
                "is_anchor",
                "ann_return",
                "ann_vol",
                "sharpe",
                "max_drawdown",
                "avg_abs_corr",
            ]
        ].round(4)
    )
    plot_selected_corr(top_selected, f"{method_name} selected-correlation heatmap")
    return stats

Candidates after start_date filter (<= 2016-01-01): 372 / 500 (dropped 128)


Resampled daily closes to weekly closes using W-FRI.
Price matrix shape before coverage filter: (522, 372) (522 weekly periods)
Coverage filter dropped 1 symbols (< 80% of trading days): ['EU']
Weekly log-return matrix shape: (521, 371)
Manual blacklist dropped 12 symbols before the statistical filters: ['ASHR', 'ECH', 'ELV', 'EWZ', 'GDXJ', 'GLD', 'KSA', 'OUNZ', 'PHYS', 'SGOL', 'UGL', 'VNM']
Total-return hurdle dropped 75 symbols below 0.2000 log total return (2016-01-01 to 2025-12-31).
Top 15 dropped: ['UVXY', 'SOXS', 'JDST', 'DUST', 'TECS', 'BOIL', 'LABD', 'SQQQ', 'VIXY', 'VXX', 'DRIP', 'FAZ', 'SRTY', 'TZA', 'SPXS']
Computing Spearman correlation matrix ...
Correlation matrix shape: (226, 226)
Shared return/correlation panel:


,value
top_n_per_method,30
return_frequency,weekly
required_anchors,"VOO, VEA"
manual_blacklist,"GLD, ASHR, ELV, KSA, EWZ, UGL, VNM, ECH, BAR, ..."
analysis_start,2016-01-01
analysis_end,2025-12-31
history_start_cutoff,2016-01-01
history_end,2025-12-31
min_total_return_hurdle,0.2000 cumulative log return
min_average_yearly_return,0.0000 average calendar-year log return


Pre-selection audit:


,step,threshold,count_before,count_after,removed_count
0,coverage,>= 80% observed return rows,372,371,1
1,blacklist,"GLD, ASHR, ELV, KSA, EWZ, UGL, VNM, ECH, BAR, ...",371,359,12
2,min_total_return,>= 0.2000 cumulative log return,359,284,75
3,min_average_yearly_return,>= 0.0000 average calendar-year log return,284,284,0
4,min_ann_vol,>= 10.0%,284,238,46
5,max_ann_vol,<= 60%,238,226,12


## Greedy Maximin

`select_by_greedy()` from `correlate_greedy.py` is seeded with the required anchors `VOO` and `VEA`. After those two names are locked in, each new ETF maximizes its minimum distance to the already selected set.

The greedy step is still:

$$
\operatorname*{argmax}_{c \notin S} \; \min_{s \in S} D(c, s)
$$

That makes it a direct low-correlation expansion around the hard anchor pair rather than around the most-liquid ETF.


In [2]:
greedy_selected, greedy_marginal = correlate_greedy.select_by_greedy(
    dist, survivors, n_select=TOP_N, anchor_tickers=ANCHOR_TICKERS
)
greedy_stats = summarize_selection("greedy", greedy_selected)

fig, ax = plt.subplots(figsize=(10, 4), constrained_layout=True)
ax.plot(
    range(2, len(greedy_marginal) + 1),
    greedy_marginal[1:],
    marker="o",
    linewidth=1.8,
    color="#1f77b4",
)
ax.set_title("Greedy marginal diversity")
ax.set_xlabel("Selection step")
ax.set_ylabel("Minimum distance to current set")
ax.grid(alpha=0.3)
display(fig)
plt.close(fig)

Greedy Maximin: fixed anchors = VOO, VEA
Selecting 30 ETFs ...
  Step  3: added IAU       (marginal min-dist = 0.6474)
  Step  4: added HYD       (marginal min-dist = 0.6229)
  Step  5: added BNO       (marginal min-dist = 0.6143)
  Step  6: added DBA       (marginal min-dist = 0.6037)
  Step  7: added XLU       (marginal min-dist = 0.5601)
  Step  8: added AMJ       (marginal min-dist = 0.5131)
  Step  9: added PGF       (marginal min-dist = 0.5071)
  Step 10: added URA       (marginal min-dist = 0.4917)
  Step 11: added KWEB      (marginal min-dist = 0.4827)
  Step 12: added EPI       (marginal min-dist = 0.4694)
  Step 13: added VWOB      (marginal min-dist = 0.4682)
  Step 14: added XBI       (marginal min-dist = 0.4664)
  Step 15: added XLP       (marginal min-dist = 0.4642)
  Step 16: added TAN       (marginal min-dist = 0.4609)
  Step 17: added JETS      (marginal min-dist = 0.4555)
  Step 18: added REM       (marginal min-dist = 0.4470)
  Step 19: added ITB       (marginal min-

,ticker,is_anchor,ann_return,ann_vol,sharpe,max_drawdown,avg_abs_corr
0,EWT,0,0.1708,0.2060,0.5863,0.3831,0.3948
1,IAU,0,0.1337,0.1455,0.5751,0.1958,0.1511
2,VOO,1,0.1430,0.1705,0.5455,0.3176,0.4924
3,XME,0,0.2134,0.3387,0.4824,0.6113,0.4244
4,ITB,0,0.1454,0.3076,0.3102,0.4994,0.3740
5,LIT,0,0.1390,0.3076,0.2893,0.6241,0.4087
6,XLV,0,0.0974,0.1687,0.2812,0.2448,0.3603
7,EPI,0,0.1026,0.1915,0.2748,0.4779,0.3424
8,URA,0,0.1502,0.3672,0.2729,0.5720,0.3415
9,XLU,0,0.0987,0.1933,0.2522,0.3313,0.2794


<Figure size 800x600 with 2 Axes>

<Figure size 1000x400 with 1 Axes>

## Anchored K-Medoids

`select_by_kmedoids()` from `correlate_kmedoids.py` treats `VOO` and `VEA` as fixed medoids, then optimizes the remaining medoids on the same signed-distance matrix.

The objective is medoid-based cluster compression on the signed-distance geometry, so each selected ETF is an actual tradable representative rather than a synthetic centroid.



In [3]:
kmedoids_selected, kmedoids_info = correlate_kmedoids.select_by_kmedoids(
    dist, survivors, n_select=TOP_N, anchor_tickers=ANCHOR_TICKERS
)
kmedoids_stats = summarize_selection("kmedoids", kmedoids_selected)

print("Anchored k-medoids cluster summary:")
display(kmedoids_info.round(4))

fig, ax = plt.subplots(figsize=(10, 4), constrained_layout=True)
ax.bar(kmedoids_info["ticker"], kmedoids_info["cluster_size"], color="#2f4b7c")
ax.set_title("Anchored k-medoids cluster sizes")
ax.set_ylabel("Assigned ETF count")
ax.tick_params(axis="x", rotation=35)
ax.grid(axis="y", alpha=0.3)
display(fig)
plt.close(fig)

  k-medoids iteration  1: updated medoids
  k-medoids iteration  2: updated medoids
  k-medoids iteration  3: updated medoids
  k-medoids iteration  4: converged
Anchored k-medoids selected 30 ETFs.
Fixed anchors: VOO, VEA
kmedoids top-30 tickers: VOO, VEA, IWS, FVD, VDE, EEM, SLV, PFF, MCHI, GUNR, EMB, XBI, VCLT, INDA, ITB, AMLP, ICLN, HYMB, XLV, ANGL, BIZD, DBA, EMLC, EWW, JETS, LIT, REM, URA, XLP, XLU


,ticker,is_anchor,ann_return,ann_vol,sharpe,max_drawdown,avg_abs_corr
0,VOO,1,0.1430,0.1705,0.5455,0.3176,0.5284
1,SLV,0,0.1582,0.2834,0.3817,0.3824,0.2208
2,ITB,0,0.1454,0.3076,0.3102,0.4994,0.3983
3,GUNR,0,0.1113,0.2083,0.2943,0.4180,0.4727
4,LIT,0,0.1390,0.3076,0.2893,0.6241,0.4341
5,XLV,0,0.0974,0.1687,0.2812,0.2448,0.3961
6,FVD,0,0.0925,0.1539,0.2763,0.3210,0.5169
7,URA,0,0.1502,0.3672,0.2729,0.5720,0.3606
8,XLU,0,0.0987,0.1933,0.2522,0.3313,0.3074
9,IWS,0,0.0974,0.2001,0.2369,0.4098,0.5608


<Figure size 800x600 with 2 Axes>

Anchored k-medoids cluster summary:


,ticker,cluster_size,avg_cluster_distance,is_anchor_medoid,selection_rank
0,VOO,47,0.1980,1,1
1,VEA,38,0.2415,1,2
2,IWS,47,0.2153,0,3
3,FVD,15,0.2756,0,4
4,VDE,14,0.2388,0,5
5,EEM,13,0.1908,0,6
6,SLV,8,0.2052,0,7
7,PFF,5,0.2315,0,8
8,MCHI,4,0.1798,0,9
9,GUNR,4,0.2618,0,10


<Figure size 1000x400 with 1 Axes>

## Maximum Diversification Ratio

`select_by_maxdiv()` from `correlate_maxdiv.py` greedily adds the ETF that maximises the equal-weight **diversification ratio** of the growing basket:

$
DR = \frac{\bar{\sigma}}{\sigma_p^{EW}} = \frac{\frac{1}{N}\sum_i \sigma_i}{\sqrt{\mathbf{w}^T \Sigma \mathbf{w}}}
$

Unlike the other methods, this directly optimises a portfolio-level diversification metric using the **full covariance structure** rather than pairwise distances or cluster membership. DR = 1 means perfect correlation (no benefit); higher is better.

Reference: Choueifaty & Coignard (2008), "Toward Maximum Diversification."


In [4]:
maxdiv_selected, maxdiv_marginal_drs = correlate_maxdiv.select_by_maxdiv(
    log_ret, survivors, n_select=TOP_N, anchor_tickers=ANCHOR_TICKERS
)
maxdiv_stats = summarize_selection("maxdiv", maxdiv_selected)

fig, ax = plt.subplots(figsize=(10, 4), constrained_layout=True)
# Skip first entry (NaN for single-asset basket)
steps = list(range(2, len(maxdiv_marginal_drs) + 1))
dr_values = maxdiv_marginal_drs[1:]
ax.plot(steps, dr_values, marker="o", linewidth=1.8, color="#2f4b7c")
ax.set_title("Max-DR: diversification ratio vs basket size")
ax.set_xlabel("Number of ETFs in basket")
ax.set_ylabel("Diversification Ratio (higher = better)")
ax.grid(alpha=0.3)
display(fig)
plt.close(fig)

  Anchors seeded: VOO, VEA
  Step   3: IAU     DR = 1.2666
  Step   4: BNO     DR = 1.4052
  Step   5: KWEB    DR = 1.4877
  Step   6: DBA     DR = 1.5612
  Step   7: VCLT    DR = 1.6220


  Step   8: DXJ     DR = 1.6486
  Step   9: GDX     DR = 1.6786
  Step  10: HYMB    DR = 1.6937
  Step  15: XLP     DR = 1.7293
  Step  20: EWH     DR = 1.7218
  Step  25: URA     DR = 1.7068
  Step  30: MCHI    DR = 1.6917
Max-DR selected 30 ETFs.
maxdiv top-30 tickers: VOO, VEA, IAU, BNO, KWEB, DBA, VCLT, DXJ, GDX, HYMB, FTGC, XBI, SPLB, KRE, XLP, HYD, PSLV, DBO, PGF, EWH, INDA, IBB, XLU, GSG, URA, EMLC, PGX, SLV, AMJ, MCHI


,ticker,is_anchor,ann_return,ann_vol,sharpe,max_drawdown,avg_abs_corr
0,IAU,0,0.1337,0.1455,0.5751,0.1958,0.2301
1,VOO,1,0.1430,0.1705,0.5455,0.3176,0.3853
2,DXJ,0,0.1403,0.2099,0.4303,0.3253,0.2537
3,SLV,0,0.1582,0.2834,0.3817,0.3824,0.2728
4,GDX,0,0.1869,0.3589,0.3814,0.4633,0.2718
5,PSLV,0,0.1500,0.2771,0.3607,0.3939,0.2722
6,URA,0,0.1502,0.3672,0.2729,0.5720,0.3019
7,XLU,0,0.0987,0.1933,0.2522,0.3313,0.2335
8,VEA,1,0.0889,0.1721,0.2262,0.3462,0.4190
9,INDA,0,0.0846,0.1864,0.1858,0.4112,0.2879


<Figure size 800x600 with 2 Axes>

<Figure size 1000x400 with 1 Axes>

## Cross-Method Summary

This last cell compares only the methods that have already been run above. It rebuilds the overlap matrix and one consensus table from the in-memory method ranks produced in this notebook.

The table is ordered by penalized average rank. If an ETF is missing from one method, that missing rank is filled with that method's `max_rank + 1`, so names that are consistently ranked well across methods rise to the top while one-off picks are penalized.



In [5]:
if not selection_results:
    raise ValueError("Run at least one selection-method cell before the summary cell.")

selection_map = {
    method_name: payload["selected"]
    for method_name, payload in selection_results.items()
}
membership = build_membership_frame(selection_map)
overlap = build_overlap_matrix(selection_map)

method_summary_rows = []
for method_name, payload in selection_results.items():
    stats = payload["stats"]
    method_summary_rows.append(
        {
            "method": method_name,
            "avg_ann_return": float(stats["ann_return"].mean()),
            "avg_ann_vol": float(stats["ann_vol"].mean()),
            "avg_sharpe": float(stats["sharpe"].mean()),
            "avg_abs_corr": float(stats["avg_abs_corr"].mean()),
        }
    )

method_summary = pd.DataFrame(method_summary_rows).sort_values(
    ["avg_sharpe", "avg_abs_corr"],
    ascending=[False, True],
)
rank_columns = [column for column in membership.columns if column.endswith("_rank")]
consensus_columns = [
    "ticker",
    "selection_count",
    *rank_columns,
]

print("Consensus table:")
display(membership.head(TOP_N)[consensus_columns].round(4))
print("Method-level summary:")
display(method_summary.round(4))

fig, axes = plt.subplots(1, 2, figsize=(16, 6), constrained_layout=True)
image = axes[0].imshow(overlap.values.astype(float), cmap="Blues")
axes[0].set_xticks(range(len(overlap.columns)), overlap.columns, rotation=30)
axes[0].set_yticks(range(len(overlap.index)), overlap.index)
axes[0].set_title(f"Top-{TOP_N} selection overlap")
for row_index in range(len(overlap.index)):
    for column_index in range(len(overlap.columns)):
        axes[0].text(
            column_index,
            row_index,
            int(overlap.iloc[row_index, column_index]),
            ha="center",
            va="center",
        )
fig.colorbar(image, ax=axes[0], fraction=0.046)

bar_colors = plt.cm.tab10(np.linspace(0, 1, len(method_summary)))
axes[1].bar(
    method_summary["method"],
    method_summary["avg_abs_corr"],
    color=bar_colors,
)
axes[1].set_title("Average absolute correlation")
axes[1].set_ylabel("Lower is better")
axes[1].grid(axis="y", alpha=0.3)

display(fig)
plt.close(fig)

Consensus table:


,ticker,selection_count,greedy_rank,kmedoids_rank,maxdiv_rank,average_rank
0,VOO,3,1.0,1.0,1.0,1.0000
1,VEA,3,2.0,2.0,2.0,2.0000
2,DBA,3,6.0,22.0,6.0,11.3333
3,IAU,2,3.0,NaN,3.0,12.3333
4,XBI,3,14.0,12.0,12.0,12.6667
5,BNO,2,5.0,NaN,4.0,13.3333
6,VCLT,3,23.0,13.0,7.0,14.3333
7,KWEB,2,11.0,NaN,5.0,15.6667
8,HYD,2,4.0,NaN,16.0,17.0000
9,HYMB,2,NaN,18.0,10.0,19.6667


Method-level summary:


,method,avg_ann_return,avg_ann_vol,avg_sharpe,avg_abs_corr
0,greedy,0.0871,0.2319,0.1526,0.3479
1,kmedoids,0.0814,0.2169,0.1292,0.3930
2,maxdiv,0.0770,0.2187,0.0960,0.2740


<Figure size 1600x600 with 3 Axes>